# 🧪 SwinIR + PaddleOCR Ablation Study (Kaggle)

**EMERGENCY MODE**: Includes 'Index Matching' (Hail Mary) if filenames don't match IDs.

## Instructions
1. **Add Data**: Upload dataset.
2. **Add Models**: Upload `best_models_for_testing`.
3. **Run All**: Compares Pretrained vs Fine-tuned SwinIR.

In [ ]:
# 📦 Install Dependencies
!pip uninstall -y numpy scipy scikit-image paddleocr paddlex
!pip install "numpy==1.26.4" "scipy==1.12.0" "scikit-image==0.22.0" "paddlepaddle-gpu==2.6.1" "paddleocr<2.8" timm transformers qwen_vl_utils rapidfuzz jiwer opencv-python-headless --no-cache-dir

In [ ]:
# 🕵️‍♂️ CONFIG
import glob
import os
import sys
from pathlib import Path
import re
import json

# Find JSON
json_candidates = glob.glob("/kaggle/input/**/textvqa_dataset.json", recursive=True)
if json_candidates:
    ANNO_PATH = Path(json_candidates[0])
    BASE_DIR = ANNO_PATH.parent
    DATA_PATH = BASE_DIR / "DATA" if (BASE_DIR / "DATA").exists() else BASE_DIR
    print(f"✅ DATA_PATH: {DATA_PATH}")
else:
    ANNO_PATH = None; DATA_PATH = None

# Find Models
swin_candidates = glob.glob("/kaggle/input/**/swinir_arch.py", recursive=True)
if swin_candidates:
    MODEL_BASE_PATH = Path(swin_candidates[0]).parent
    sys.path.append(str(MODEL_BASE_PATH))
else:
    MODEL_BASE_PATH = Path("ERROR")

# 📂 PRE-LOAD ALL IMAGES (For Index Matching)
SORTED_IMAGES = []
if DATA_PATH:
    # Get all jpg/pngs
    exts = ['*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG']
    files = []
    for e in exts:
        files.extend(glob.glob(str(DATA_PATH / e)))
    # Sort them to align with JSON (hopefully)
    SORTED_IMAGES = sorted(files)
    print(f"✅ Pre-loaded {len(SORTED_IMAGES)} images for fallback.")

# 🧩 SMART FILE MAPPER
FILE_MAP = {}
if DATA_PATH:
    for f in SORTED_IMAGES:
        fname = os.path.basename(f)
        FILE_MAP[fname] = f
        digits = "".join(filter(str.isdigit, fname))
        if len(digits) > 8: FILE_MAP[digits] = f

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
from paddleocr import PaddleOCR

# Import SwinIR
try: from swinir_arch import SwinIR
except: pass

PRETRAINED_SWINIR_PATH = MODEL_BASE_PATH / "swinir_medium_pretrained_bsrgan.pth"
FINETUNED_SWINIR_PATH = MODEL_BASE_PATH / "swinir_medium_textzoom_curriculum.pth"

def load_swinir(model_path):
    args = {'upscale': 4, 'in_chans': 3, 'img_size': 64, 'window_size': 8, 'img_range': 1.0, 'depths': [6]*6, 'embed_dim': 180, 'num_heads': [6]*6, 'mlp_ratio': 2.0, 'upsampler': 'nearest+conv', 'resi_connection': '1conv'}
    model = SwinIR(**args)
    ckpt = torch.load(model_path, map_location='cuda')
    model.load_state_dict(ckpt['params_ema'] if 'params_ema' in ckpt else ckpt['params'] if 'params' in ckpt else ckpt, strict=True)
    return model.cuda().eval()

def load_paddle():
    det = glob.glob("/kaggle/input/**/inference_det_finetuned", recursive=True)
    rec = glob.glob("/kaggle/input/**/inference_rec_finetuned", recursive=True)
    return PaddleOCR(det_model_dir=det[0] if det else None, rec_model_dir=rec[0] if rec else None, use_gpu=True, lang='en', show_log=False)

def swinir_inference(model, img_path):
    img_path = str(img_path)
    img_lq = cv2.imread(img_path, cv2.IMREAD_COLOR).astype(np.float32) / 255.
    img_lq = np.transpose(img_lq if img_lq.shape[2] == 3 else img_lq[:, :, [2, 1, 0]], (2, 0, 1)) 
    img_lq = torch.from_numpy(img_lq).float().unsqueeze(0).cuda()
    window_size = 8
    _, _, h, w = img_lq.size()
    h_pad = (h // window_size + 1) * window_size - h
    w_pad = (w // window_size + 1) * window_size - w
    img_lq = torch.cat([img_lq, torch.flip(img_lq, [2])], 2)[:, :, :h + h_pad, :]
    img_lq = torch.cat([img_lq, torch.flip(img_lq, [3])], 3)[:, :, :, :w + w_pad]
    with torch.no_grad(): output = model(img_lq)
    output = output[..., :h * 4, :w * 4]
    output = output.data.squeeze().float().cpu().clamp_(0, 1).numpy()
    if output.ndim == 3: output = np.transpose(output[[2, 1, 0], :, :], (1, 2, 0)) 
    return (output * 255.0).round().astype(np.uint8)

def calculate_anls(pred, gt_list):
    from rapidfuzz import fuzz
    if not pred and not gt_list: return 1.0
    best = 0.0
    for gt in gt_list:
        sim = fuzz.ratio(str(pred).lower(), str(gt).lower())/100.0
        best = max(best, 0.0 if sim < 0.5 else sim)
    return best

In [ ]:
# 🚀 MAIN EVALUATION (INDEX MATCH)
def run_ablation(name, swin_model, paddle_tool, samples):
    print(f"\n📢 START: {name}")
    scores = []
    missing = 0
    
    for i, sample in enumerate(tqdm(samples)):
        img_id = sample['image_id'] 
        
        # --- INTELLIGENT MATCHING ---
        p = None
        # 1. Map
        if img_id in FILE_MAP: p = FILE_MAP[img_id]
        # 2. Digits
        elif "".join(filter(str.isdigit, img_id)) in FILE_MAP: p = FILE_MAP["".join(filter(str.isdigit, img_id))]
        
        # 3. EMERGENCY FALLBACK: INDEX MATCHING
        # If we can't find it by name, assume simple 1-to-1 sorted order
        if not p and i < len(SORTED_IMAGES):
            p = Path(SORTED_IMAGES[i])
            if i == 0: print(f"⚠️ MATCH FAIL for '{img_id}'. Using index {i} file: '{p.name}'")
        
        if not p: 
            missing += 1
            continue
            
        try:
            sr_bgr = swinir_inference(swin_model, p)
            cv2.imwrite("/kaggle/working/temp.jpg", sr_bgr)
            ocr_res = paddle_tool.ocr("/kaggle/working/temp.jpg", cls=False)
            text = "".join([l[1][0] for l in ocr_res[0]]) if ocr_res and ocr_res[0] else ""
            scores.append(calculate_anls(text, sample.get('answers', [])))
        except: pass
        
        if (i+1)%50==0: 
             m = np.mean(scores) if scores else 0
             print(f"[{i+1}] Mean: {m:.4f}")

    m = np.mean(scores) if scores else 0
    print(f"🏆 {name}: {m:.4f} (Evaluated {len(scores)} samples)\n")
    return m

if ANNO_PATH:
    with open(ANNO_PATH) as f: samples = json.load(f)['data']
    paddle_tool = load_paddle()
    
    swin_pt = load_swinir(PRETRAINED_SWINIR_PATH)
    s1 = run_ablation("Run 1: Pretrained", swin_pt, paddle_tool, samples)
    
    swin_ft = load_swinir(FINETUNED_SWINIR_PATH)
    s2 = run_ablation("Run 2: Finetuned", swin_ft, paddle_tool, samples)
    
    print(f"FINAL:\nPT: {s1:\nFT: {s2}")